# Cosmos DB vector search without LangChain

This notebook uses the direct Azure SDK pattern: generate an embedding, query Cosmos DB vector search, and send the retrieved product context to Azure OpenAI in a manual RAG flow.

In [ ]:
import json
import os
import time

from dotenv import load_dotenv
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
from openai import AzureOpenAI

In [ ]:
loaded = load_dotenv()
if not loaded:
    raise RuntimeError(".env file could not be loaded.")

key_vault_url = os.environ["KEY_VAULT_URL"]
secret_name = os.environ["COSMOS_DB_CONNECTION_STRING_SECRET_NAME"]

credential = DefaultAzureCredential()
secret_client = SecretClient(vault_url=key_vault_url, credential=credential)
connection_string = secret_client.get_secret(secret_name).value

if connection_string is None:
    raise RuntimeError(f"Key Vault secret '{secret_name}' has no value.")

client = CosmosClient.from_connection_string(connection_string)
 
database_name = "test_db_5"
source_container_name = "product"
target_container_name = "product_v"

if source_container_name == target_container_name:
    raise ValueError("Source and target containers must be different.")

database = client.get_database_client(database_name)
source_products_container = database.get_container_client(source_container_name)
source_products_container.read()

aoai_endpoint = os.environ["AOAI_ENDPOINT"]
api_version = os.environ["AOAI_API_VERSION"]
aoai_key = os.environ["AOAI_KEY"]
embeddings_deployment_name = os.environ["EMBEDDINGS_DEPLOYMENT_NAME"]

ai_client = AzureOpenAI(
    azure_endpoint=aoai_endpoint,
    api_version=api_version,
    api_key=aoai_key,
)

In [ ]:
vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/contentVector",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": 1536,
        }
    ]
}

indexing_policy = {
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [
        {"path": "/\"_etag\"/?"},
        {"path": "/contentVector/*"},
    ],
    "vectorIndexes": [{"path": "/contentVector", "type": "diskANN"}],
}

product_v_container = database.create_container_if_not_exists(
    id=target_container_name,
    partition_key=PartitionKey(path="/categoryId"),
    indexing_policy=indexing_policy,
    vector_embedding_policy=vector_embedding_policy,
)

target_properties = product_v_container.read()
if target_properties.get("partitionKey", {}).get("paths") != ["/categoryId"]:
    raise ValueError("The target container must use /categoryId as its partition key.")
if target_properties.get("vectorEmbeddingPolicy") != vector_embedding_policy:
    raise ValueError("Choose a new target container with the same vector policy before generating embeddings.")

In [ ]:
def generate_embeddings(text: str):
    if not text.strip():
        raise ValueError("Embedding input must contain text.")

    response = ai_client.embeddings.create(
        input=text, model=embeddings_deployment_name, encoding_format="float"
    )
    embeddings = response.data[0].embedding

    expected_dimensions = vector_embedding_policy["vectorEmbeddings"][0]["dimensions"]
    if len(embeddings) != expected_dimensions:
        raise ValueError(
            f"Embedding has {len(embeddings)} dimensions; the container policy expects "
            f"{expected_dimensions}."
        )

    time.sleep(0.5)
    return embeddings


def vector_search(container, prompt: str, vector_field_name: str = "contentVector", num_results: int = 5):
    query_embedding = generate_embeddings(prompt)
    items = container.query_items(
        query=f"""SELECT TOP @num_results itm.id, itm.categoryId, VectorDistance(itm.{vector_field_name}, @embedding) AS SimilarityScore 
                FROM itm
                ORDER BY VectorDistance(itm.{vector_field_name}, @embedding)
                """,
        parameters=[
            {"name": "@num_results", "value": num_results},
            {"name": "@embedding", "value": query_embedding},
        ],
        enable_cross_partition_query=True,
    )
    return list(items)

# Read existing Cosmos DB products, generate embeddings, and persist enriched copies.
# Reruns upsert the same id/categoryId pairs rather than creating duplicates.
print("API version:", api_version)
print("Deployment:", embeddings_deployment_name)
print("Dimensions:", len(generate_embeddings("test product")))

written = 0
skipped = 0
products = source_products_container.query_items(
    query="SELECT * FROM c", enable_cross_partition_query=True
)
for product in products:
    if not product.get("id") or not product.get("categoryId"):
        raise ValueError("Every source product needs an id and categoryId.")
    text = "\n".join(
        f"{field}: {product[field]}"
        for field in ("name", "description", "categoryName")
        if product.get(field) and str(product[field]).strip()
    )
    if not text:
        skipped += 1
        print(f"Skipping {product['id']}: no name, description, or categoryName.")
        continue
    # Cosmos DB supplies these system properties; don't copy them to the target.
    document = {
        key: value for key, value in product.items()
        if key not in {"_rid", "_self", "_etag", "_attachments", "_ts", "contentVector"}
    }
    document["contentVector"] = generate_embeddings(text)
    product_v_container.upsert_item(body=document)
    written += 1
    print(f"Stored {document['id']} ({len(document['contentVector'])} dimensions)")

if written == 0:
    raise RuntimeError(
        f"No products were written. Check {database_name}/{source_container_name} "
        f"contains products with text. Skipped {skipped} products."
    )
print(f"Finished: wrote {written} products to {target_container_name}; skipped {skipped}.")

In [ ]:
# Example query
search_text = "lightweight product for sport"
results = vector_search(product_v_container, search_text, num_results=5)

print(f"Top {len(results)} matches for: {search_text!r}")
for rank, item in enumerate(results, start=1):
    print(f"{rank}. id={item['id']} | categoryId={item['categoryId']} | distance={item['SimilarityScore']:.4f}")

In [ ]:
system_prompt = """You are a helpful product assistant. Use only the product details supplied in context. If the answer is not present, say you do not know."""
deployment_name = os.environ.get("CHAT_DEPLOYMENT_NAME") or os.environ.get("AOAI_CHAT_DEPLOYMENT_NAME") or embeddings_deployment_name

def query_item_by_id(container, item_id: str, category_id: str):
    doc = container.read_item(item=item_id, partition_key=category_id)
    doc.pop("contentVector", None)
    return doc

def rag_with_vector_search(container, prompt: str, num_results: int = 5):
    matches = vector_search(container, prompt, num_results=num_results)
    product_context = ""

    for result in matches:
        product = query_item_by_id(container, result["id"], result["categoryId"])
        product_context += json.dumps(product, default=str, indent=2) + "\n\n"

    formatted_prompt = system_prompt + "\n\nContext:\n" + product_context + "\n\nUser question:\n" + prompt
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": formatted_prompt},
    ]

    completion = ai_client.chat.completions.create(messages=messages, model=deployment_name)
    return completion.choices[0].message.content

answer = rag_with_vector_search(product_v_container, "Which product is best for hiking in rough terrain?")
print(answer)